In [ ]:
import yfinance as yf

{'address1': 'One Microsoft Way', 'city': 'Redmond', 'state': 'WA', 'zip': '98052-6399', 'country': 'United States', 'phone': '425 882 8080', 'website': 'https://www.microsoft.com', 'industry': 'Software - Infrastructure', 'industryKey': 'software-infrastructure', 'industryDisp': 'Software - Infrastructure', 'sector': 'Technology', 'sectorKey': 'technology', 'sectorDisp': 'Technology', 'longBusinessSummary': 'Microsoft Corporation develops and supports software, services, devices, and solutions worldwide. The Productivity and Business Processes segment offers Microsoft 365 commercial, enterprise mobility + security, windows commercial, power BI, exchange, sharepoint, Microsoft teams, security and compliance, and copilot; Microsoft 365 commercial products, such as Windows commercial on-premises and office licensed services; Microsoft 365 consumer products and cloud services, including Microsoft 365 consumer subscriptions, office licensed on-premises, and other consumer services; LinkedI

In [6]:
import pandas as pd

# Top 10 US stocks by market cap (typical order; use yfinance for live P/E)
# Sources: major indices / market cap rankings (NVDA, AAPL, MSFT, GOOGL, AMZN, META, AVGO, TSLA, BRK-B, LLY)
tickers = ["NVDA", "AAPL", "MSFT", "GOOGL", "AMZN", "META", "AVGO", "TSLA", "BRK-B", "LLY"]
print("Top 10 US stocks by market cap (tickers):", tickers)

Top 10 US stocks by market cap (tickers): ['NVDA', 'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'AVGO', 'TSLA', 'BRK-B', 'LLY']


In [7]:
# Get P/E ratio for each ticker using yfinance
pe_data = []
for ticker in tickers:
    t = yf.Ticker(ticker)
    info = t.info
    name = info.get("shortName", ticker)
    trailing_pe = info.get("trailingPE")
    forward_pe = info.get("forwardPE")
    pe_data.append({
        "Ticker": ticker,
        "Company": name,
        "P/E (trailing)": trailing_pe,
        "P/E (forward)": forward_pe,
    })

df_pe = pd.DataFrame(pe_data)
df_pe

,Ticker,Company,P/E (trailing),P/E (forward)
0,NVDA,NVIDIA Corporation,45.893566,24.035738
1,AAPL,Apple Inc.,35.205063,29.974737
2,MSFT,Microsoft Corporation,25.118347,21.257565
3,GOOGL,Alphabet Inc.,29.839186,24.257076
4,AMZN,"Amazon.com, Inc.",29.333334,22.675670
5,META,"Meta Platforms, Inc.",28.171211,18.656930
6,AVGO,Broadcom Inc.,69.941180,23.179405
7,TSLA,"Tesla, Inc.",384.214940,143.738220
8,BRK-B,Berkshire Hathaway Inc. New,16.258880,21.003431
9,LLY,Eli Lilly and Company,46.087982,25.424217


In [ ]:
# Quarterly revenue: same-quarter-over-same-quarter YoY growth (last 5 years)
from datetime import datetime

def get_quarterly_revenue_growth(ticker_symbol, years=5):
    """Same quarter YoY revenue growth for last 5 years. Returns DataFrame with Q1–Q4 growths."""
    t = yf.Ticker(ticker_symbol)
    qis = t.quarterly_income_stmt
    if qis is None or qis.empty:
        return None
    # Find revenue row (Yahoo uses different labels)
    rev_row = None
    for label in ["Total Revenue", "Revenue", "TotalRevenue", "Operating Revenue"]:
        if label in qis.index:
            rev_row = qis.loc[label]
            break
    if rev_row is None:
        return None
    rev = rev_row.dropna()
    rev = rev[rev != 0]
    rev = rev.sort_index()
    # Keep last 5 years of quarters (~20 quarters)
    cutoff = datetime.now().replace(year=datetime.now().year - years)
    rev = rev[rev.index >= pd.Timestamp(cutoff)]
    # Build (year, quarter) -> revenue
    by_yq = {}
    for dt in rev.index:
        y, q = dt.year, (dt.month - 1) // 3 + 1
        by_yq[(y, q)] = rev[dt]
    # Same-quarter YoY growth: (this_q - same_q_last_year) / same_q_last_year
    growths = []
    for (y, q), val in sorted(by_yq.items()):
        prev = by_yq.get((y - 1, q))
        if prev is not None and prev != 0:
            g = (val - prev) / prev * 100
            growths.append({"Year": y, "Quarter": f"Q{q}", "Revenue_YoY_%": round(g, 2)})
    return pd.DataFrame(growths) if growths else None

# Collect for all tickers and show (e.g. last 8 quarters of growth for readability)
quarterly_growth_all = {}
for ticker in tickers:
    df_q = get_quarterly_revenue_growth(ticker, years=5)
    if df_q is not None:
        quarterly_growth_all[ticker] = df_q

# Combine into one table: Ticker, Year, Quarter, Revenue_YoY_%
rows = []
for ticker in tickers:
    if ticker in quarterly_growth_all:
        df = quarterly_growth_all[ticker].copy()
        df.insert(0, "Ticker", ticker)
        rows.append(df)
df_quarterly_growth = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
print("Quarterly revenue growth (same quarter YoY, %) – last 5 years")
df_quarterly_growth

In [ ]:
# Annual EPS and YoY EPS growth (last 5 years)
def get_annual_eps_growth(ticker_symbol, years=5):
    """Annual EPS and YoY EPS growth for last 5 years."""
    t = yf.Ticker(ticker_symbol)
    ann = t.income_stmt
    if ann is None or ann.empty:
        return None
    eps_series = None
    for label in ["Diluted EPS", "DilutedEps", "Basic EPS", "BasicEps", "Earnings Per Share"]:
        if label in ann.index:
            eps_series = ann.loc[label].dropna()
            break
    if eps_series is None and "Net Income" in ann.index and "Diluted Average Shares" in ann.index:
        ni = ann.loc["Net Income"].dropna()
        sh = ann.loc["Diluted Average Shares"].dropna()
        common = ni.index.intersection(sh.index)
        if len(common) > 0:
            eps_series = (ni.reindex(common).ffill().bfill() / sh.reindex(common).ffill().bfill())
    if eps_series is None or eps_series.empty:
        return None
    eps_series = eps_series.sort_index()
    cutoff = datetime.now().replace(year=datetime.now().year - years)
    eps_series = eps_series[eps_series.index >= pd.Timestamp(cutoff)].tail(years + 1)
    # Build year -> EPS (take latest report per fiscal year)
    by_year = {}
    for dt in eps_series.index:
        by_year[dt.year] = eps_series[dt]
    out = []
    for y in sorted(by_year.keys()):
        eps = by_year[y]
        prev_eps = by_year.get(y - 1)
        g = (eps - prev_eps) / prev_eps * 100 if prev_eps and prev_eps != 0 else None
        out.append({"Year": y, "EPS": round(eps, 4), "EPS_YoY_%": round(g, 2) if g is not None else None})
    return pd.DataFrame(out) if out else None

eps_growth_all = {}
for ticker in tickers:
    df_eps = get_annual_eps_growth(ticker, years=5)
    if df_eps is not None:
        df_eps.insert(0, "Ticker", ticker)
        eps_growth_all[ticker] = df_eps

rows_eps = list(eps_growth_all.values())
df_annual_eps_growth = pd.concat(rows_eps, ignore_index=True) if rows_eps else pd.DataFrame()
print("Annual EPS and YoY EPS growth (%) – last 5 years")
df_annual_eps_growth